In [ ]:
# Standard library
import json
import random
import time
from argparse import ArgumentParser

# Third-party
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.profilers import AdvancedProfiler

# First-party
from neural_lam import constants, utils, config
from neural_lam.weather_dataset import WeatherDataset
from neural_lam.downscaling_dataset import DownscalingDataset
from neural_lam.netCDF_dataset import NetCDFDataset
from neural_lam.models.graph_efm import GraphEFM
from neural_lam.models.graph_fm import GraphFM
from neural_lam.models.graphcast import GraphCast
from neural_lam.models.diffusion import Diffusion
from neural_lam.models.ir_sde import IR_SDE
from neural_lam.models.stochastic_interpolants import SI

In [ ]:
config_loader = config.Config.from_file('neural_lam/clim_config.yaml')

In [ ]:
train_dataset = NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    )

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=1,
)

In [ ]:
import random

idx = random.randint(0, len(train_dataset) - 1)
test_batch = train_dataset[idx]

test_batch['LQ'].shape, test_batch['ESM'].shape, test_batch['HQ'].shape

In [ ]:
ESM = test_batch['ESM'][config_loader.dataset.downscaling_idx,...]
LQ = test_batch['LQ'][config_loader.dataset.downscaling_idx,...]
HQ = test_batch['HQ']

ESM.shape, LQ.shape, HQ.shape

In [ ]:
import matplotlib.pyplot as plt

def plot_channels(tensor, title):
    tensor = tensor.detach().cpu()  # if it's a torch tensor

    C = tensor.shape[0]

    fig, axes = plt.subplots(1, C, figsize=(4*C, 4))
    
    if C == 1:
        axes = [axes]

    for i in range(C):
        im = axes[i].imshow(tensor[i], cmap='plasma', origin="lower")
        axes[i].set_title(f"{title} - Channel {i}")
        axes[i].axis('off')
        
        # Add colorbar for each subplot
        fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
    
plot_channels(ESM, "ESM")
plot_channels(LQ, "LQ")
plot_channels(HQ, "HQ")

In [ ]:
import torch

def channel_stats(tensor):
    tensor = tensor.detach().cpu()
    
    means = tensor.mean(dim=(1, 2))   # (C,)
    sums  = tensor.sum(dim=(1, 2))    # (C,)
    
    return means, sums

esm_mean, esm_sum = channel_stats(ESM)
lq_mean, lq_sum = channel_stats(LQ)
hq_mean, hq_sum   = channel_stats(HQ)

In [ ]:
for i in range(len(esm_mean)):
    print(f"Channel {i}:")
    print(f"  Mean  -> ESM: {esm_mean[i]:.6f}, LQ: {lq_mean[i]:.6f}, HQ: {hq_mean[i]:.6f}, Diff: {(esm_mean[i]-hq_mean[i]):.6e}")
    print(f"  Sum   -> ESM: {esm_sum[i]:.6f}, LQ: {lq_sum[i]:.6f},  HQ: {hq_sum[i]:.6f},  Diff: {(esm_sum[i]-hq_sum[i]):.6e}")

In [ ]:
import matplotlib.pyplot as plt

def plot_value_distribution(t1, t2, title):
    t1 = t1.detach().cpu().flatten()
    t2 = t2.detach().cpu().flatten()
    
    plt.hist(t1, bins=50, alpha=0.5, label='ESM')
    plt.hist(t2, bins=50, alpha=0.5, label='HQ')
    
    plt.title(title)
    plt.legend()
    plt.show()
    
    
plot_value_distribution(ESM, HQ, "Pixel Value Distribution (All Channels)")

In [ ]:
def plot_per_channel_distribution(t1, t2):
    t1 = t1.detach().cpu()
    t2 = t2.detach().cpu()
    
    C = t1.shape[0]
    
    for i in range(C):
        plt.figure()
        plt.hist(t1[i].flatten(), bins=50, alpha=0.5, label='ESM')
        plt.hist(t2[i].flatten(), bins=50, alpha=0.5, label='HQ')
        plt.title(f"Channel {i} Distribution")
        plt.legend()
        plt.show()
        
plot_per_channel_distribution(LQ, HQ)

In [ ]:
import torch.nn.functional as F

def downsample_bicubic(HQ, target_size):
    # HQ: (C, H, W)
    HQ = HQ.unsqueeze(0)  # add batch dim → (1, C, H, W)

    down = F.interpolate(
        HQ,
        size=target_size,
        mode='bicubic',
        align_corners=False
    )

    return down.squeeze(0)

HQ_bicubic = downsample_bicubic(HQ, (40, 55))

In [ ]:
import torch.nn.functional as F

def downsample_sum_pool(HQ, kernel_size):
    # HQ: (C, H, W)
    HQ = HQ.unsqueeze(0)  # (1, C, H, W)

    down = F.avg_pool2d(
        HQ,
        kernel_size=kernel_size,
        stride=kernel_size
    )

    # convert avg → sum
    down = down * (kernel_size[0] * kernel_size[1])

    return down.squeeze(0)

HQ_sum = downsample_sum_pool(HQ, (10, 10))

In [ ]:
def compute_metrics(pred, target):
    mse = ((pred - target) ** 2).mean()
    mae = (pred - target).abs().mean()
    return mse.item(), mae.item()

In [ ]:
HQ_bicubic.shape, ESM.shape, HQ_sum.shape

In [ ]:
mse_bicubic, mae_bicubic = compute_metrics(HQ_bicubic, ESM)
mse_sum, mae_sum = compute_metrics(HQ_sum, ESM)

print("Bicubic  -> MSE:", mse_bicubic, "MAE:", mae_bicubic)
print("SumPool  -> MSE:", mse_sum, "MAE:", mae_sum)

In [ ]:
def per_channel_metrics(pred, target):
    C = pred.shape[0]
    for i in range(C):
        mse = ((pred[i] - target[i])**2).mean()
        mae = (pred[i] - target[i]).abs().mean()
        print(f"Channel {i}: MSE={mse:.6f}, MAE={mae:.6f}")
        
per_channel_metrics(HQ_bicubic, ESM)
per_channel_metrics(HQ_sum, ESM)

In [ ]:
diff = HQ_bicubic - ESM

plot_channels(ESM, "ESM")

plot_channels(HQ_bicubic, "Bicubic")

diff = HQ_sum - ESM
plot_channels(HQ_sum, "SumPool")